In [2]:
# read files from bronze layer
from pyspark.sql import functions as F
from pyspark.sql.types import *

# read csv files with schema infered
df_customers = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("Files/bronze/customers.csv")

df_policies = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("Files/bronze/policies.csv")

df_claims = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("Files/bronze/claims.csv")

print("✓ Bronze data loaded")
print(f"Customers: {df_customers.count()} rows")
print(f"Policies: {df_policies.count()} rows")
print(f"Claims: {df_claims.count()} rows")

StatementMeta(, 58773712-f2ce-49c5-ac74-04dd9ff5600a, 4, Finished, Available, Finished)

✓ Bronze data loaded
Customers: 50 rows
Policies: 50 rows
Claims: 50 rows


In [3]:
display(df_customers.limit(10))

StatementMeta(, 58773712-f2ce-49c5-ac74-04dd9ff5600a, 5, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 74a2f603-205f-40a1-aa06-965dac6b44a4)

In [4]:
display(df_claims.limit(10))

StatementMeta(, 58773712-f2ce-49c5-ac74-04dd9ff5600a, 6, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 61ae112d-8881-46c8-94f6-f383f86229f6)

In [5]:
display(df_policies.limit(10))

StatementMeta(, 58773712-f2ce-49c5-ac74-04dd9ff5600a, 7, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 8495db11-71bd-4a9c-9c65-a89ce4f66bab)

## Clean customer data

In [10]:
df_customers_clean = (
    df_customers
    .dropna(subset=["cust_id"])
    .withColumn("cust_id", col("cust_id").cast("int"))
    .withColumn("CustomerName", initcap(trim(col("Full Name"))))
    .withColumn("Date_of_Birth", to_date(col("Date_of_Birth")))
    .withColumn("Contact", when(col("Contact").isNull(), "Not Provided").otherwise(col("Contact")))
    .drop("Full Name")
    .dropDuplicates()
)
display(df_customers_clean)

StatementMeta(, 58773712-f2ce-49c5-ac74-04dd9ff5600a, 12, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 89b7c699-7d20-4963-aa87-0e5c186f00db)

## Cleaning the policies data

In [12]:
display(df_policies.limit(7))

StatementMeta(, 58773712-f2ce-49c5-ac74-04dd9ff5600a, 14, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 3bbe683c-9d46-47c1-9472-74f8e5df35e7)

## 

In [13]:
df_policies_clean = (
    df_policies
    .withColumn("policy_Type", initcap(trim(col("policy_Type"))))
    .withColumn("status", initcap(trim(col("status"))))
    .withColumn("Start_Date", to_date(col("Start_Date"), "dd/MM/yyyy"))
    .withColumn("Coverage_Amount", when(col("Coverage_Amount") == "not available", None)
                .otherwise(col("Coverage_Amount").cast("double")))
    .withColumn("cust_id", col("cust_id").cast("int"))
    .dropna(subset=["cust_id", "policy_id"])
)
display(df_policies_clean)

StatementMeta(, 58773712-f2ce-49c5-ac74-04dd9ff5600a, 15, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, e39f4e71-0ec6-473d-b5e9-a6cf5da3a5c4)

## Cleaning the claims data


In [15]:
display(df_claims.limit(5))

StatementMeta(, 58773712-f2ce-49c5-ac74-04dd9ff5600a, 17, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 956bcd3b-d449-47b1-bc21-985994dee36b)

In [16]:
df_claims_clean = (
    df_claims
    .withColumn("claim_date", to_date(col("claim_date")))
    .withColumn("claim_amount", col("claim_amount").cast("double"))
    .withColumn("status", initcap(trim(col("status"))))
    .dropna(subset=["policy_id"])
)
display(df_claims_clean)

StatementMeta(, 58773712-f2ce-49c5-ac74-04dd9ff5600a, 18, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, abd1afbe-63a8-4ca4-82fd-606e6773bd88)

##  Save as Delta Tables (Silver)

In [17]:
df_customers_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_customers")

StatementMeta(, 58773712-f2ce-49c5-ac74-04dd9ff5600a, 19, Finished, Available, Finished)

In [18]:
df_policies_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_policies")

StatementMeta(, 58773712-f2ce-49c5-ac74-04dd9ff5600a, 20, Finished, Available, Finished)

In [19]:
df_claims_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_claims")

StatementMeta(, 58773712-f2ce-49c5-ac74-04dd9ff5600a, 21, Finished, Available, Finished)

In [20]:
print("Silver tables created successfully!")
print("  - silver_customers")
print("  - silver_policies")
print("  - silver_claims")

StatementMeta(, 58773712-f2ce-49c5-ac74-04dd9ff5600a, 22, Finished, Available, Finished)

Silver tables created successfully!
  - silver_customers
  - silver_policies
  - silver_claims


## 

In [21]:
%%sql 
select * from insurance_lakehouse.silver_customers

StatementMeta(, 58773712-f2ce-49c5-ac74-04dd9ff5600a, 23, Finished, Available, Finished)

<Spark SQL result set with 50 rows and 5 fields>